# Lightweight Spatiotemporal-Multimodal Hand Gesture Recognition
## SLRF-Net + MATS-Fuse + UCR-Gate — Full TensorFlow implementation

This notebook follows the eight stages in the supplied research outline. It uses **real user-provided / downloaded data only**; it never creates synthetic training samples or substitutes dummy labels.

### Dataset expectations
- Primary image dataset: HaGRID / HaGRIDv2, organized in subject-preserving `train`, `val` (or `validation`), and `test` partitions. Class subfolders are supported.
- An annotation JSON parser is included for common record layouts. If your HaGRID download has a different layout, inspect the printed audit and adjust `IMAGE_ROOT` / `ANNOTATION_JSON` in the configuration cell.
- Optional dynamic data: a CSV manifest with `sequence_id, frame_path, label, subject_id` columns. A still-image dataset alone cannot validate temporal motion recognition; without a sequence manifest, the notebook explicitly runs the static / image-geometric pathway and marks temporal sequence experiments as unavailable.
- Optional additional evaluation datasets (EHWGesture / MultiModal Hand Gesture / Leap Motion) can be loaded using the same folder or manifest interface only after their labels and subject partitions are mapped.

**Important:** The document names YOLOv13, GenENet, L-SHADE and GNN as existing methods, but does not provide their architectures or checkpoints. This implementation compares practical reproducible baselines (MobileNetV3, HOG/LBP + SVM, MLP, and the proposed modules) and does not falsely label those baselines as exact reproductions of the named methods.

## 0. Environment and reproducibility

In [ ]:
# Run in Google Colab. Restart runtime only if dependency installation requests it.
!pip -q install opencv-python-headless scikit-image scikit-learn pandas matplotlib seaborn scipy statsmodels tqdm joblib

In [ ]:
import os, json, glob, time, math, random, hashlib, warnings, gc, pathlib, shutil
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report, matthews_corrcoef, cohen_kappa_score,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    top_k_accuracy_score, balanced_accuracy_score)
from sklearn.manifold import TSNE
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import joblib

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 1. Configuration — set your real dataset paths

In [ ]:
# Google Drive is optional. Set DATA_ROOT to the directory containing train/val/test.
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print("Drive mount skipped:", e)

# CHANGE THIS to your extracted HaGRIDv2 dataset directory.
DATA_ROOT = "/content/drive/MyDrive/HaGRIDv2"
IMAGE_SIZE = (160, 160)
BATCH_SIZE = 32
EPOCHS_BACKBONE = 8
EPOCHS_PROPOSED = 25
LEARNING_RATE = 1e-4
LATENT_DIM = 128
MAX_IMAGES_PER_CLASS = None  # e.g. 500 for a manageable real-data subset; None = all
OUTPUT_DIR = "/content/SLRF_MATS_UCR_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Optional annotation JSON. Leave None for class-folder datasets.
ANNOTATION_JSON = None
# Optional dynamic sequence manifest CSV. Required columns:
# sequence_id, frame_path, label, subject_id
SEQUENCE_MANIFEST = None

# Optional independent evaluation dataset in class-folder format.
EXTERNAL_TEST_ROOT = None

print("DATA_ROOT exists:", os.path.isdir(DATA_ROOT), DATA_ROOT)

## 2. Stage 1 — Dataset discovery, annotation parsing and audit

In [ ]:
IMG_EXTS = {'.jpg','.jpeg','.png','.bmp','.webp'}

def find_partition_dir(root, names):
    if not root or not os.path.isdir(root): return None
    for n in names:
        p = os.path.join(root, n)
        if os.path.isdir(p): return p
    return None

def discover_class_folder(partition_dir):
    rows=[]
    if not partition_dir: return pd.DataFrame(columns=['path','label','subject_id','partition'])
    for cls in sorted(os.listdir(partition_dir)):
        cp=os.path.join(partition_dir,cls)
        if not os.path.isdir(cp): continue
        files=[]
        for r,_,fs in os.walk(cp):
            files.extend(os.path.join(r,f) for f in fs if pathlib.Path(f).suffix.lower() in IMG_EXTS)
        if MAX_IMAGES_PER_CLASS is not None: files=files[:MAX_IMAGES_PER_CLASS]
        for p in files:
            rows.append({'path':p,'label':cls,'subject_id':None})
    return pd.DataFrame(rows)

def _walk_annotation_records(obj, base_dir, out):
    # Flexible recursive extraction of common annotation record schemas.
    if isinstance(obj, dict):
        path_keys = ['path','image_path','filename','file_name','image','img']
        label_keys = ['label','gesture','class','class_name','category']
        path_val = next((obj[k] for k in path_keys if k in obj and isinstance(obj[k],str)), None)
        label_val = next((obj[k] for k in label_keys if k in obj and isinstance(obj[k],str)), None)
        if path_val and label_val:
            p=path_val if os.path.isabs(path_val) else os.path.join(base_dir,path_val)
            if os.path.isfile(p):
                out.append({'path':p,'label':label_val,
                            'subject_id':obj.get('user_id',obj.get('subject_id'))})
        for v in obj.values(): _walk_annotation_records(v,base_dir,out)
    elif isinstance(obj, list):
        for v in obj: _walk_annotation_records(v,base_dir,out)

def load_annotation_json(json_path, image_base):
    with open(json_path,'r',encoding='utf-8') as f: obj=json.load(f)
    out=[]; _walk_annotation_records(obj,image_base,out)
    return pd.DataFrame(out).drop_duplicates(subset=['path']) if out else pd.DataFrame()

def audit_partition(df, name):
    print(f"\\n{name}: {len(df):,} images")
    if len(df)==0: return
    print("Classes:", df.label.nunique(), "| missing files:", (~df.path.map(os.path.isfile)).sum())
    print(df.label.value_counts().head(20).to_string())
    print("Subjects known:", df.subject_id.notna().sum())

train_dir=find_partition_dir(DATA_ROOT,['train','Train'])
val_dir=find_partition_dir(DATA_ROOT,['val','valid','validation','Val'])
test_dir=find_partition_dir(DATA_ROOT,['test','Test'])

if ANNOTATION_JSON and os.path.isfile(ANNOTATION_JSON):
    ann=load_annotation_json(ANNOTATION_JSON, DATA_ROOT)
    if len(ann):
        # Annotation-only layouts should have explicit partition folders or a partition column;
        # never randomly split images when subject-separated partitions are supplied.
        print("Annotation records found:",len(ann))
        print("Use the partition-specific annotation files if your JSON stores split metadata.")
    else:
        print("No supported image/label records found in JSON. Check its schema.")

train_df=discover_class_folder(train_dir)
val_df=discover_class_folder(val_dir)
test_df=discover_class_folder(test_dir)
for d,n in [(train_df,'TRAIN'),(val_df,'VALIDATION'),(test_df,'TEST')]: audit_partition(d,n)

if len(train_df)==0 or len(test_df)==0:
    raise FileNotFoundError(
        "Could not find class-folder train and test partitions. Set DATA_ROOT to the extracted "
        "HaGRIDv2 root, or adapt discover_class_folder() to your annotation layout. "
        "Do not create a random image split if the dataset supplies subject-wise partitions."
    )

# Validation partition is required for calibration and model selection.
if len(val_df)==0:
    raise FileNotFoundError("A separate validation partition is required for calibration. Locate the original HaGRID validation split.")
classes=sorted(train_df.label.unique())
if set(val_df.label)-set(classes) or set(test_df.label)-set(classes):
    raise ValueError("Validation/test contains labels absent from training. Reconcile class mapping explicitly.")
class_to_id={c:i for i,c in enumerate(classes)}
id_to_class={i:c for c,i in class_to_id.items()}
for d in (train_df,val_df,test_df):
    d['y']=d.label.map(class_to_id).astype('int32')
print("Number of gesture classes:",len(classes))

## 3. Stage 1 continued — duplicate, image-size and bounding-box quality checks

In [ ]:
def perceptual_hash(path, hash_size=8):
    im=cv2.imread(path,cv2.IMREAD_GRAYSCALE)
    if im is None: return None
    im=cv2.resize(im,(hash_size*4,hash_size*4))
    d=cv2.dct(np.float32(im))
    low=d[:hash_size,:hash_size]
    med=np.median(low[1:])
    bits=low>med
    return ''.join('1' if b else '0' for b in bits.flatten())

def image_quality_audit(df, sample_limit=5000):
    sample=df.head(sample_limit)
    dims=[]; hashes=[]; bad=[]
    for p in tqdm(sample.path,desc='Image audit'):
        im=cv2.imread(p)
        if im is None: bad.append(p); continue
        dims.append(im.shape[:2])
        hashes.append(perceptual_hash(p))
    valid_hash=[h for h in hashes if h is not None]
    dup_count=len(valid_hash)-len(set(valid_hash))
    print("Unreadable images:",len(bad),"| sampled dimensions:",Counter(dims).most_common(5))
    print("Exact perceptual-hash duplicates in sample:",dup_count)
    return {'bad_files':bad,'duplicate_hash_count':dup_count,'dimensions':dims}

quality_report=image_quality_audit(pd.concat([train_df,val_df,test_df],ignore_index=True))

## 4. Stage 2 — ROI extraction, compact image preprocessing and geometric descriptors

In [ ]:
def crop_roi(image, bbox=None):
    h,w=image.shape[:2]
    if bbox is not None:
        # bbox assumed normalized xywh or xyxy only when explicitly adapted by user.
        x1,y1,x2,y2=bbox
        x1=max(0,min(w-1,int(x1*w))); x2=max(x1+1,min(w,int(x2*w)))
        y1=max(0,min(h-1,int(y1*h))); y2=max(y1+1,min(h,int(y2*h)))
        image=image[y1:y2,x1:x2]
    return image

def load_rgb(path, size=IMAGE_SIZE):
    im=cv2.imread(path)
    if im is None: raise ValueError("Unreadable image: "+path)
    im=cv2.cvtColor(im,cv2.COLOR_BGR2RGB)
    im=cv2.resize(im,size,interpolation=cv2.INTER_AREA)
    return im

def hog_lbp_descriptor(rgb):
    from skimage.feature import hog, local_binary_pattern
    gray=cv2.cvtColor(rgb,cv2.COLOR_RGB2GRAY)
    h=hog(gray,orientations=9,pixels_per_cell=(16,16),cells_per_block=(2,2),
          block_norm='L2-Hys',feature_vector=True)
    lbp=local_binary_pattern(gray, P=8, R=1, method='uniform')
    hist,_=np.histogram(lbp.ravel(),bins=np.arange(0,11),density=True)
    return np.concatenate([h.astype('float32'),hist.astype('float32')])

# Optional MediaPipe landmark extraction. If unavailable, geometric descriptors use image gradients
# and a validity flag; the notebook does not fabricate hand landmarks.
try:
    import mediapipe as mp
    mp_hands=mp.solutions.hands
    HANDS_AVAILABLE=True
    hands_detector=mp_hands.Hands(static_image_mode=True,max_num_hands=1,
        min_detection_confidence=0.4,model_complexity=0)
except Exception as e:
    HANDS_AVAILABLE=False
    hands_detector=None
    print("MediaPipe unavailable; fallback geometric descriptors enabled:",str(e)[:180])

def geometric_descriptor(rgb):
    if HANDS_AVAILABLE:
        result=hands_detector.process(rgb)
        if result.multi_hand_landmarks:
            pts=np.array([[p.x,p.y,p.z] for p in result.multi_hand_landmarks[0].landmark],dtype=np.float32)
            pts[:,:2]-=pts[0,:2]
            scale=np.linalg.norm(pts[:,0:2],axis=1).max()+1e-6
            pts[:,:2]/=scale
            return np.concatenate([pts.flatten(),[1.0]]).astype('float32')
    # Fallback: compact edge-orientation / contour geometry, not a claimed landmark vector.
    gray=cv2.cvtColor(rgb,cv2.COLOR_RGB2GRAY)
    edges=cv2.Canny(gray,60,140)
    moments=cv2.moments(edges)
    hu=cv2.HuMoments(moments).flatten()
    hu=np.sign(hu)*np.log1p(np.abs(hu)*1e6)
    return np.concatenate([hu.astype('float32'),np.zeros(63,dtype='float32'),[0.0]]).astype('float32')

# Cache descriptors so expensive image processing is performed once.
CACHE_DIR=os.path.join(OUTPUT_DIR,'feature_cache'); os.makedirs(CACHE_DIR,exist_ok=True)
def extract_handcrafted(df, split):
    out_path=os.path.join(CACHE_DIR,f'{split}_hoggeo.npz')
    if os.path.isfile(out_path):
        z=np.load(out_path,allow_pickle=True)
        return z['hog'],z['geo']
    hs=[]; gs=[]
    for p in tqdm(df.path,desc=f'{split} HOG/geometry'):
        rgb=load_rgb(p)
        hs.append(hog_lbp_descriptor(rgb))
        gs.append(geometric_descriptor(rgb))
    hs=np.asarray(hs,dtype='float32'); gs=np.asarray(gs,dtype='float32')
    np.savez_compressed(out_path,hog=hs,geo=gs)
    return hs,gs

X_hog_train,G_train=extract_handcrafted(train_df,'train')
X_hog_val,G_val=extract_handcrafted(val_df,'val')
X_hog_test,G_test=extract_handcrafted(test_df,'test')
print("HOG:",X_hog_train.shape,"Geometry:",G_train.shape)

## 5. Stage 2 continued — frozen MobileNetV3-Small visual representation

In [ ]:
AUTOTUNE=tf.data.AUTOTUNE
preprocess=tf.keras.applications.mobilenet_v3.preprocess_input

def make_image_ds(df, training=False):
    paths=df.path.astype(str).values
    labels=df.y.values.astype('int32')
    ds=tf.data.Dataset.from_tensor_slices((paths,labels))
    def _load(path,label):
        raw=tf.io.read_file(path)
        im=tf.io.decode_image(raw,channels=3,expand_animations=False)
        im=tf.image.resize(im,IMAGE_SIZE)
        im=tf.cast(im,tf.float32)
        if training:
            im=tf.image.random_flip_left_right(im)
            im=tf.image.random_brightness(im,0.08)
            im=tf.image.random_contrast(im,0.9,1.1)
        return preprocess(im),label
    ds=ds.map(_load,num_parallel_calls=AUTOTUNE)
    if training: ds=ds.shuffle(min(len(df),10000),seed=SEED,reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds=make_image_ds(train_df,True)
val_ds=make_image_ds(val_df,False)
test_ds=make_image_ds(test_df,False)

backbone=tf.keras.applications.MobileNetV3Small(
    input_shape=(*IMAGE_SIZE,3),include_top=False,weights='imagenet',pooling='avg')
backbone.trainable=False
inp=tf.keras.Input(shape=(*IMAGE_SIZE,3))
x=backbone(inp,training=False)
x=tf.keras.layers.Dropout(0.2)(x)
logits=tf.keras.layers.Dense(len(classes),name='baseline_logits')(x)
baseline_model=tf.keras.Model(inp,logits,name='MobileNetV3Small_baseline')
baseline_model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=['accuracy'])
history_baseline=baseline_model.fit(train_ds,validation_data=val_ds,epochs=EPOCHS_BACKBONE,verbose=1,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=3,restore_best_weights=True)])

## 6. Stage 3 — SLRF-Net: sparse low-rank projection and prototype classifier

In [ ]:
class SparseLowRank(tf.keras.layers.Layer):
    def __init__(self, latent_dim=128, groups=8, l1_strength=1e-5, group_strength=1e-4, **kwargs):
        super().__init__(**kwargs); self.latent_dim=latent_dim; self.groups=groups
        self.l1_strength=l1_strength; self.group_strength=group_strength
    def build(self,input_shape):
        self.proj=self.add_weight(name='low_rank_projection',
            shape=(int(input_shape[-1]),self.latent_dim),initializer='glorot_uniform',trainable=True)
        self.bias=self.add_weight(name='bias',shape=(self.latent_dim,),initializer='zeros',trainable=True)
    def call(self,x):
        z=tf.matmul(x,self.proj)+self.bias
        self.add_loss(self.l1_strength*tf.reduce_mean(tf.abs(self.proj)))
        # Group-lasso over contiguous latent dimensions.
        chunks=tf.split(z,self.groups,axis=-1) if self.latent_dim%self.groups==0 else [z]
        group_penalty=tf.add_n([tf.sqrt(tf.reduce_mean(tf.square(c))+1e-8) for c in chunks])
        self.add_loss(self.group_strength*group_penalty)
        return tf.math.l2_normalize(tf.nn.relu(z),axis=-1)

class SLRFNet(tf.keras.Model):
    def __init__(self, feature_dim, n_classes, latent_dim=128):
        super().__init__()
        self.proj=SparseLowRank(latent_dim=latent_dim)
        self.classifier=tf.keras.layers.Dense(n_classes)
    def call(self,x,training=False):
        z=self.proj(x)
        return self.classifier(z)

# Extract frozen visual embeddings. This makes low-rank learning fast and repeatable.
embedding_model=tf.keras.Model(backbone.input,backbone.output)
def embed_dataset(df, split):
    f=os.path.join(CACHE_DIR,f'{split}_mobilenet_embeddings.npy')
    if os.path.isfile(f): return np.load(f)
    ds=make_image_ds(df,False).map(lambda x,y:x)
    arr=embedding_model.predict(ds,verbose=1)
    np.save(f,arr); return arr

V_train=embed_dataset(train_df,'train')
V_val=embed_dataset(val_df,'val')
V_test=embed_dataset(test_df,'test')
y_train=train_df.y.to_numpy(); y_val=val_df.y.to_numpy(); y_test=test_df.y.to_numpy()

slrf=SLRFNet(V_train.shape[1],len(classes),LATENT_DIM)
slrf.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=['accuracy'])
hist_slrf=slrf.fit(V_train,y_train,validation_data=(V_val,y_val),
    epochs=EPOCHS_PROPOSED,batch_size=BATCH_SIZE,verbose=1,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=5,restore_best_weights=True)])

## 7. Stage 3 continued — class prototypes, cosine scores and optional PCA baseline

In [ ]:
def get_latent(model, X):
    return model.proj(X,training=False).numpy()
Z_train=get_latent(slrf,V_train); Z_val=get_latent(slrf,V_val); Z_test=get_latent(slrf,V_test)
prototypes=np.vstack([Z_train[y_train==c].mean(axis=0) for c in range(len(classes))])
prototypes=prototypes/(np.linalg.norm(prototypes,axis=1,keepdims=True)+1e-8)

def prototype_logits(Z, protos, temperature=10.0):
    Z=Z/(np.linalg.norm(Z,axis=1,keepdims=True)+1e-8)
    return temperature*(Z@protos.T)
proto_val=prototype_logits(Z_val,prototypes)
proto_test=prototype_logits(Z_test,prototypes)
print("Prototype score shape:",proto_test.shape)

## 8. Stage 4 — dynamic gesture sequences (only when real sequence manifest is supplied)

In [ ]:
sequence_data_available=False
if SEQUENCE_MANIFEST and os.path.isfile(SEQUENCE_MANIFEST):
    seq_df=pd.read_csv(SEQUENCE_MANIFEST)
    required={'sequence_id','frame_path','label','subject_id'}
    if not required.issubset(seq_df.columns):
        raise ValueError(f"Sequence manifest must contain columns: {sorted(required)}")
    seq_df['frame_path']=seq_df.frame_path.apply(lambda p:p if os.path.isabs(p) else os.path.join(DATA_ROOT,p))
    seq_df=seq_df[seq_df.frame_path.map(os.path.isfile)].copy()
    seq_df=seq_df[seq_df.label.isin(class_to_id)].copy()
    seq_df['y']=seq_df.label.map(class_to_id).astype(int)
    sequence_data_available=len(seq_df)>0
else:
    seq_df=pd.DataFrame()
    print("No sequence manifest supplied. Dynamic motion claims/metrics will be skipped; static geometric fusion remains available.")

def build_sequences(manifest, max_len=16):
    Xs=[]; Xg=[]; ys=[]; ids=[]
    for sid,g in tqdm(manifest.groupby('sequence_id'),desc='Build real sequences'):
        g=g.sort_values('frame_path')
        if len(g)<2: continue
        frames=[]; geos=[]
        for p in g.frame_path:
            rgb=load_rgb(p)
            frames.append(rgb.astype('float32')/255.)
            geos.append(geometric_descriptor(rgb))
        # Uniform temporal resampling, preserving actual frame order.
        idx=np.linspace(0,len(frames)-1,min(max_len,len(frames))).astype(int)
        frames=np.asarray([frames[i] for i in idx])
        geos=np.asarray([geos[i] for i in idx])
        if len(frames)<max_len:
            pad=max_len-len(frames)
            frames=np.concatenate([np.repeat(frames[-1:],pad,axis=0),frames],axis=0)
            geos=np.concatenate([np.repeat(geos[-1:],pad,axis=0),geos],axis=0)
        Xs.append(frames); Xg.append(geos); ys.append(int(g.y.iloc[0])); ids.append(sid)
    return np.asarray(Xs),np.asarray(Xg),np.asarray(ys),np.asarray(ids)

if sequence_data_available:
    X_seq,G_seq,y_seq,seq_ids=build_sequences(seq_df)
    print("Sequence tensors:",X_seq.shape,G_seq.shape,y_seq.shape)
else:
    X_seq=G_seq=y_seq=seq_ids=None

## 9. Stage 5 — MATS-Fuse: low-rank multimodal fusion, reliability gates and temporal smoothing

In [ ]:
# Static multimodal pathway: visual MobileNet embeddings + geometric descriptors.
# Fit normalization on training data only.
geo_scaler=StandardScaler()
Gtr=geo_scaler.fit_transform(G_train)
Gva=geo_scaler.transform(G_val)
Gte=geo_scaler.transform(G_test)
vis_scaler=StandardScaler()
Vtr_s=vis_scaler.fit_transform(V_train)
Vva_s=vis_scaler.transform(V_val)
Vte_s=vis_scaler.transform(V_test)

# Compress high-dimensional visual and geometric modalities separately.
vis_pca=PCA(n_components=min(128,Vtr_s.shape[1],len(Vtr_s)-1),random_state=SEED)
geo_pca=PCA(n_components=min(32,Gtr.shape[1],len(Gtr)-1),random_state=SEED)
Vtr_p=vis_pca.fit_transform(Vtr_s); Vva_p=vis_pca.transform(Vva_s); Vte_p=vis_pca.transform(Vte_s)
Gtr_p=geo_pca.fit_transform(Gtr); Gva_p=geo_pca.transform(Gva); Gte_p=geo_pca.transform(Gte)

class MATSFuse(tf.keras.Model):
    def __init__(self,n_classes,latent=128):
        super().__init__()
        self.v_proj=tf.keras.layers.Dense(latent,activation='relu')
        self.g_proj=tf.keras.layers.Dense(latent,activation='relu')
        self.v_rel=tf.keras.layers.Dense(1,activation='sigmoid',name='visual_reliability')
        self.g_rel=tf.keras.layers.Dense(1,activation='sigmoid',name='geometric_reliability')
        self.fuse=tf.keras.layers.Dense(latent,activation='relu')
        self.drop=tf.keras.layers.Dropout(.25)
        self.out=tf.keras.layers.Dense(n_classes)
    def call(self,inputs,training=False):
        v,g=inputs
        zv=self.v_proj(v); zg=self.g_proj(g)
        rv=self.v_rel(v); rg=self.g_rel(g)
        den=rv+rg+1e-7
        wv=rv/den; wg=rg/den
        fused=self.fuse(wv*zv+wg*zg)
        fused=self.drop(fused,training=training)
        return self.out(fused)

mats=MATSFuse(len(classes),LATENT_DIM)
mats.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=['accuracy'])
hist_mats=mats.fit((Vtr_p,Gtr_p),y_train,validation_data=((Vva_p,Gva_p),y_val),
    epochs=EPOCHS_PROPOSED,batch_size=BATCH_SIZE,verbose=1,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=5,restore_best_weights=True)])

# Reliability weights are model-estimated; evaluate modality-only and fusion variants.
def mats_components(model,V,G):
    zv=model.v_proj(V).numpy(); zg=model.g_proj(G).numpy()
    rv=model.v_rel(V).numpy().ravel(); rg=model.g_rel(G).numpy().ravel()
    den=rv+rg+1e-7
    return rv/den,rg/den,zv,zg
wvis,wgeo,_,_=mats_components(mats,Vte_p,Gte_p)
print("Mean visual/geometric reliability:",float(wvis.mean()),float(wgeo.mean()))

# Optional temporal branch: TSM-like channel shift + compact GRU on actual sequences.
if sequence_data_available:
    seq_in=tf.keras.Input(shape=(X_seq.shape[1],X_seq.shape[2],X_seq.shape[3],3))
    # Frame-wise compact encoder, then temporal aggregation.
    frame_encoder=tf.keras.Sequential([
        tf.keras.layers.TimeDistributed(tf.keras.layers.Rescaling(1./255.)),
        tf.keras.layers.TimeDistributed(tf.keras.layers.Conv2D(16,3,strides=2,activation='relu')),
        tf.keras.layers.TimeDistributed(tf.keras.layers.GlobalAveragePooling2D())])
    q=frame_encoder(seq_in)
    # Temporal shift proxy: concatenate previous/next shifted feature channels.
    q_prev=tf.concat([tf.zeros_like(q[:,:1,:]),q[:,:-1,:]],axis=1)
    q_next=tf.concat([q[:,1:,:],tf.zeros_like(q[:,:1,:])],axis=1)
    q=tf.keras.layers.Concatenate()([q,q_prev,q_next])
    q=tf.keras.layers.GRU(64)(q)
    seq_out=tf.keras.layers.Dense(len(classes))(q)
    temporal_model=tf.keras.Model(seq_in,seq_out)
    temporal_model.compile(optimizer='adam',loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=['accuracy'])
    print("Temporal model created. Train/evaluate using sequence-level subject splits before reporting dynamic results.")
else:
    temporal_model=None

## 10. Stage 6 — UCR-Gate: temperature scaling, conformal sets, entropy and hysteresis

In [ ]:
# Use validation logits for temperature calibration; test data remains untouched.
val_logits=mats.predict((Vva_p,Gva_p),verbose=0)
test_logits=mats.predict((Vte_p,Gte_p),verbose=0)

class TemperatureScaler(tf.keras.Model):
    def __init__(self):
        super().__init__(); self.log_temp=tf.Variable(0.0,trainable=True,dtype=tf.float32)
    def call(self,logits): return logits/tf.exp(self.log_temp)
temp_scaler=TemperatureScaler()
opt=tf.keras.optimizers.Adam(.03)
y_val_tf=tf.constant(y_val,dtype=tf.int32)
for _ in range(300):
    with tf.GradientTape() as tape:
        scaled=temp_scaler(tf.constant(val_logits,dtype=tf.float32))
        loss=tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(y_val_tf,scaled,from_logits=True))
    grads=tape.gradient(loss,[temp_scaler.log_temp]); opt.apply_gradients(zip(grads,[temp_scaler.log_temp]))
temperature=float(tf.exp(temp_scaler.log_temp).numpy())
print("Learned temperature:",temperature)

def softmax_np(logits):
    z=logits-np.max(logits,axis=1,keepdims=True)
    e=np.exp(z); return e/e.sum(axis=1,keepdims=True)
p_val=softmax_np(val_logits/temperature)
p_test=softmax_np(test_logits/temperature)

# Split-conformal calibration: finite-sample quantile of 1 - p_true.
alpha=0.10
scores=1.0-p_val[np.arange(len(y_val)),y_val]
q_level=min(1.0,math.ceil((len(scores)+1)*(1-alpha))/len(scores))
qhat=float(np.quantile(scores,q_level,method='higher'))
def conformal_sets(probs,q):
    return [np.where(1.0-row<=q)[0].tolist() for row in probs]
pred_sets=conformal_sets(p_test,qhat)
set_sizes=np.array([len(s) for s in pred_sets])
coverage=np.mean([int(y_test[i]) in pred_sets[i] for i in range(len(y_test))])
print(f"Conformal q={qhat:.4f} | empirical test coverage={coverage:.4f} | mean set size={set_sizes.mean():.3f}")

entropy=-(p_test*np.log(p_test+1e-12)).sum(axis=1)
max_conf=p_test.max(axis=1)
pred=p_test.argmax(axis=1)
# Prototype distance is used as an additional out-of-distribution indicator.
proto_dist=1.0-(Z_test/(np.linalg.norm(Z_test,axis=1,keepdims=True)+1e-8)) @ prototypes.T
nearest_proto_dist=proto_dist.min(axis=1)

# Validation-selected thresholds, not hand-picked on test labels.
v_entropy=-(p_val*np.log(p_val+1e-12)).sum(axis=1)
v_conf=p_val.max(axis=1)
v_proto=1.0-(Z_val/(np.linalg.norm(Z_val,axis=1,keepdims=True)+1e-8)) @ prototypes.T
v_dist=v_proto.min(axis=1)
conf_thr=float(np.quantile(v_conf,0.10))
ent_thr=float(np.quantile(v_entropy,0.90))
dist_thr=float(np.quantile(v_dist,0.95))

accepted=(max_conf>=conf_thr)&(entropy<=ent_thr)&(nearest_proto_dist<=dist_thr)&(set_sizes==1)
gated_pred=np.where(accepted,pred,-1)  # -1 means no_gesture / uncertain rejection
print("Accepted fraction:",accepted.mean(),"| rejected fraction:",1-accepted.mean())

def hysteresis_control(confidences, activate=.80, deactivate=.60):
    active=False; states=[]
    for c in confidences:
        if not active and c>=activate: active=True
        elif active and c<deactivate: active=False
        states.append(active)
    return np.asarray(states,dtype=bool)

## 11. Stage 7 — evaluation metrics, calibration and unknown-gesture rejection

In [ ]:
def classification_metrics(y_true, logits, name, class_names=classes):
    probs=softmax_np(logits); yhat=probs.argmax(axis=1)
    pr,rc,f1,_=precision_recall_fscore_support(y_true,yhat,average='macro',zero_division=0)
    cm=confusion_matrix(y_true,yhat,labels=np.arange(len(class_names)))
    specificity=[]
    for i in range(len(class_names)):
        tp=cm[i,i]; fn=cm[i,:].sum()-tp; fp=cm[:,i].sum()-tp
        tn=cm.sum()-tp-fn-fp
        specificity.append(tn/(tn+fp+1e-12))
    result={'Model':name,'Accuracy':accuracy_score(y_true,yhat),'Precision_macro':pr,
        'Recall_macro':rc,'Macro_F1':f1,'Specificity_macro':np.mean(specificity),
        'MCC':matthews_corrcoef(y_true,yhat),'Cohen_Kappa':cohen_kappa_score(y_true,yhat),
        'Top1':accuracy_score(y_true,yhat),
        'Top5':top_k_accuracy_score(y_true,probs,k=min(5,len(class_names)),labels=np.arange(len(class_names)))}
    return result,probs,yhat,cm

baseline_logits=baseline_model.predict(test_ds,verbose=0)
slrf_logits=slrf.predict(V_test,verbose=0)
mats_logits=test_logits
rows=[]; predictions={}
for name,lg in [('MobileNetV3Small',baseline_logits),('SLRF-Net',slrf_logits),('MATS-Fuse',mats_logits)]:
    r,p,yh,cm=classification_metrics(y_test,lg,name)
    rows.append(r); predictions[name]=(p,yh,cm)
metrics_df=pd.DataFrame(rows)
display(metrics_df.round(4))
metrics_df.to_csv(os.path.join(OUTPUT_DIR,'classification_metrics.csv'),index=False)

# Gated metrics: report coverage and selective accuracy separately.
selective_acc=accuracy_score(y_test[accepted],pred[accepted]) if accepted.any() else np.nan
print("UCR-Gate selective accuracy:",selective_acc)
print("UCR-Gate coverage / activation rate:",accepted.mean())
print("UCR-Gate conformal coverage:",coverage,"mean set size:",set_sizes.mean())
print(classification_report(y_test,pred,target_names=classes,zero_division=0))

## 12. Stage 7 continued — ROC, PR, confusion matrix, reliability and class-wise results

In [ ]:
# Multi-class ROC and PR curves (one-vs-rest).
Y_bin=label_binarize(y_test,classes=np.arange(len(classes)))
probs=softmax_np(mats_logits)
plt.figure(figsize=(9,7))
for i,c in enumerate(classes):
    if Y_bin[:,i].sum()==0 or Y_bin[:,i].sum()==len(Y_bin): continue
    fpr,tpr,_=roc_curve(Y_bin[:,i],probs[:,i])
    plt.plot(fpr,tpr,label=f'{c} (AUC={auc(fpr,tpr):.3f})')
plt.plot([0,1],[0,1],'k--'); plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('MATS-Fuse Multi-class ROC'); plt.legend(fontsize=7,bbox_to_anchor=(1.02,1),loc='upper left')
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'multiclass_roc.png'),dpi=180); plt.show()

plt.figure(figsize=(9,7))
for i,c in enumerate(classes):
    if Y_bin[:,i].sum()==0: continue
    prc,rec,_=precision_recall_curve(Y_bin[:,i],probs[:,i])
    ap=average_precision_score(Y_bin[:,i],probs[:,i])
    plt.plot(rec,prc,label=f'{c} (AP={ap:.3f})')
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('MATS-Fuse Precision–Recall')
plt.legend(fontsize=7,bbox_to_anchor=(1.02,1),loc='upper left')
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'multiclass_pr.png'),dpi=180); plt.show()

cm=predictions['MATS-Fuse'][2]
fig,ax=plt.subplots(figsize=(max(8,len(classes)*.45),max(7,len(classes)*.4)))
im=ax.imshow(cm,cmap='Blues'); ax.set_title('MATS-Fuse Confusion Matrix')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_xticks(range(len(classes))); ax.set_yticks(range(len(classes)))
ax.set_xticklabels(classes,rotation=90,fontsize=7); ax.set_yticklabels(classes,fontsize=7)
fig.colorbar(im,ax=ax); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'confusion_matrix.png'),dpi=180); plt.show()

def reliability_bins(probs,y,n_bins=15):
    conf=probs.max(axis=1); correct=(probs.argmax(axis=1)==y)
    edges=np.linspace(0,1,n_bins+1); acc=[]; cbar=[]; counts=[]
    for a,b in zip(edges[:-1],edges[1:]):
        m=(conf>=a)&(conf<(b if b<1 else b+1e-9))
        if m.any(): acc.append(correct[m].mean()); cbar.append(conf[m].mean()); counts.append(m.sum())
    ece=np.sum(np.asarray(counts)/len(y)*np.abs(np.asarray(acc)-np.asarray(cbar)))
    return np.asarray(cbar),np.asarray(acc),float(ece)
raw_probs=softmax_np(mats_logits)
cal_probs=p_test
c1,a1,ece1=reliability_bins(raw_probs,y_test); c2,a2,ece2=reliability_bins(cal_probs,y_test)
plt.figure(figsize=(6,6)); plt.plot([0,1],[0,1],'k--',label='Perfect')
plt.plot(c1,a1,'o-',label=f'Before scaling ECE={ece1:.3f}')
plt.plot(c2,a2,'s-',label=f'After scaling ECE={ece2:.3f}')
plt.xlabel('Confidence'); plt.ylabel('Accuracy'); plt.title('Reliability Diagram'); plt.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'reliability_diagram.png'),dpi=180); plt.show()

per_class=classification_report(y_test,pred,target_names=classes,output_dict=True,zero_division=0)
pd.DataFrame(per_class).T.to_csv(os.path.join(OUTPUT_DIR,'classwise_report.csv'))
print("ECE before/after:",ece1,ece2)

## 13. Stage 7 continued — model size, batch-one latency, throughput and quantization

In [ ]:
def profile_keras_model(model, sample, warmup=20, runs=100):
    x=sample[:1]
    for _ in range(warmup): model(x,training=False)
    t0=time.perf_counter()
    for _ in range(runs): model(x,training=False)
    elapsed=(time.perf_counter()-t0)/runs
    return {'latency_ms':elapsed*1000,'throughput_fps':1.0/elapsed}

def save_temp_and_size(model,filename):
    p=os.path.join(OUTPUT_DIR,filename)
    model.save(p)
    total=sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(p) for f in fs)
    return total/1024**2

latency={}
latency['baseline']=profile_keras_model(baseline_model,tf.convert_to_tensor(np.zeros((1,*IMAGE_SIZE,3),dtype='float32')))
latency['SLRF-Net']=profile_keras_model(slrf,tf.convert_to_tensor(V_test[:1]))
latency['MATS-Fuse']=profile_keras_model(mats,(tf.convert_to_tensor(Vte_p[:1]),tf.convert_to_tensor(Gte_p[:1])))
print("Latency profile:",latency)

# TensorFlow Lite INT8 conversion of the image baseline.
def representative_dataset():
    for xb,yb in train_ds.unbatch().batch(1).take(100):
        yield [tf.cast(xb,tf.float32)]
try:
    converter=tf.lite.TFLiteConverter.from_keras_model(baseline_model)
    converter.optimizations=[tf.lite.Optimize.DEFAULT]
    converter.representative_dataset=representative_dataset
    converter.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type=tf.int8
    converter.inference_output_type=tf.int8
    tflite_model=converter.convert()
    tflite_path=os.path.join(OUTPUT_DIR,'mobilenet_baseline_int8.tflite')
    with open(tflite_path,'wb') as f: f.write(tflite_model)
    print("INT8 TFLite size (MB):",len(tflite_model)/1024**2)
except Exception as e:
    print("INT8 conversion unavailable for this runtime/model:",repr(e))

## 14. Stage 8 — ablation experiments and robustness evaluation

In [ ]:
# Ablations on the same held-out test set. All variants use real extracted features.
def fit_simple_head(Xtr,Xva,Xte,ytr,yva,epochs=12):
    model=tf.keras.Sequential([tf.keras.layers.Input(shape=(Xtr.shape[1],)),
        tf.keras.layers.Dense(128,activation='relu'),tf.keras.layers.Dropout(.2),
        tf.keras.layers.Dense(len(classes))])
    model.compile(optimizer='adam',loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=['accuracy'])
    model.fit(Xtr,ytr,validation_data=(Xva,yva),epochs=epochs,batch_size=BATCH_SIZE,verbose=0,
        callbacks=[tf.keras.callbacks.EarlyStopping(patience=3,restore_best_weights=True)])
    return model

ablation_models={}
ablation_models['Visual only']=fit_simple_head(Vtr_p,Vva_p,Vte_p,y_train,y_val)
ablation_models['Geometry only']=fit_simple_head(Gtr_p,Gva_p,Gte_p,y_train,y_val)
ablation_models['Concatenation']=fit_simple_head(np.c_[Vtr_p,Gtr_p],np.c_[Vva_p,Gva_p],np.c_[Vte_p,Gte_p],y_train,y_val)
ablation_rows=[]
for name,m in ablation_models.items():
    lg=m.predict(Vte_p if name=='Visual only' else Gte_p if name=='Geometry only' else np.c_[Vte_p,Gte_p],verbose=0)
    r,_,_,_=classification_metrics(y_test,lg,name); ablation_rows.append(r)
r,_,_,_=classification_metrics(y_test,mats_logits,'MATS-Fuse'); ablation_rows.append(r)
ablation_df=pd.DataFrame(ablation_rows)
display(ablation_df.round(4))
ablation_df.to_csv(os.path.join(OUTPUT_DIR,'ablation_metrics.csv'),index=False)

# Robustness: evaluate deterministic image corruptions on the held-out test images.
def corrupt_images(df, mode):
    arr=[]
    for p in tqdm(df.path,desc=mode):
        im=load_rgb(p)
        if mode=='brightness_low': im=np.clip(im.astype(np.float32)*0.55,0,255).astype('uint8')
        elif mode=='brightness_high': im=np.clip(im.astype(np.float32)*1.35,0,255).astype('uint8')
        elif mode=='blur': im=cv2.GaussianBlur(im,(9,9),2)
        elif mode=='occlusion':
            im=im.copy(); h,w=im.shape[:2]; im[h//3:2*h//3,w//3:2*w//3]=0
        arr.append(preprocess(im.astype('float32')))
    return np.asarray(arr,dtype='float32')

robust_rows=[]
for mode in ['brightness_low','brightness_high','blur','occlusion']:
    Xc=corrupt_images(test_df,mode)
    # Image baseline measures robustness directly at pixel level.
    lg=baseline_model.predict(Xc,batch_size=BATCH_SIZE,verbose=0)
    rr,_,_,_=classification_metrics(y_test,lg,mode)
    robust_rows.append(rr)
robust_df=pd.DataFrame(robust_rows)
display(robust_df[['Model','Accuracy','Macro_F1']].round(4))
robust_df.to_csv(os.path.join(OUTPUT_DIR,'robustness_metrics.csv'),index=False)

## 15. Statistical significance — repeated seeds, ANOVA and Tukey HSD

In [ ]:
# Statistical tests require independent repeated runs/folds. A single test score is not a sample.
# This cell uses repeated stratified validation splits of the training partition for a lightweight
# MLP comparison. Subject grouping should be used instead if subject IDs are present.
from sklearn.model_selection import StratifiedKFold
X_stat=np.c_[Vtr_p,Gtr_p]
skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
fold_scores=[]
for fold,(it,iv) in enumerate(skf.split(X_stat,y_train),1):
    mm=fit_simple_head(X_stat[it],X_stat[iv],X_stat[iv],y_train[it],y_train[iv],epochs=8)
    pp=mm.predict(X_stat[iv],verbose=0).argmax(axis=1)
    fold_scores.append({'fold':fold,'macro_f1':precision_recall_fscore_support(y_train[iv],pp,average='macro',zero_division=0)[2]})
fold_df=pd.DataFrame(fold_scores)
display(fold_df)
print("Fold mean ± std:",fold_df.macro_f1.mean(),fold_df.macro_f1.std(ddof=1))
# One-way ANOVA / Tukey only become meaningful with repeated independent scores per model.
# Current five-fold sample is shown for the MLP alone; do not claim pairwise significance from it.

## 16. Explainability — Grad-CAM and SHAP-compatible feature importance

In [ ]:
# Grad-CAM for the image baseline. Find the final convolution layer automatically.
def find_last_conv_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer,tf.keras.Model):
            found=find_last_conv_layer(layer)
            if found: return found
        if isinstance(layer,tf.keras.layers.Conv2D): return layer.name
    return None

def gradcam(model, image_batch, class_index=None):
    conv_name=find_last_conv_layer(model)
    if conv_name is None:
        raise ValueError("No Conv2D layer found for Grad-CAM.")
    # The convolutional backbone is a nested model inside the classifier.
    backbone_layer=model.get_layer('MobileNetV3Small')
    conv_layer=backbone_layer.get_layer(conv_name)
    grad_model=tf.keras.Model(model.inputs,[conv_layer.output,model.output])
    with tf.GradientTape() as tape:
        conv_out,preds=grad_model(image_batch)
        idx=tf.argmax(preds[0]) if class_index is None else class_index
        score=preds[:,idx]
    grads=tape.gradient(score,conv_out)
    weights=tf.reduce_mean(grads,axis=(1,2))
    cam=tf.reduce_sum(weights[:,None,None,:]*conv_out,axis=-1)
    cam=tf.nn.relu(cam)[0]
    cam=cam/(tf.reduce_max(cam)+1e-8)
    return cam.numpy(),int(idx.numpy())

sample_path=test_df.path.iloc[0]
sample_rgb=load_rgb(sample_path)
sample_tensor=tf.convert_to_tensor(preprocess(sample_rgb.astype('float32'))[None,...])
try:
    heat,cls_idx=gradcam(baseline_model,sample_tensor)
    heat=cv2.resize(heat,(IMAGE_SIZE[1],IMAGE_SIZE[0]))
    overlay=cv2.applyColorMap(np.uint8(255*heat),cv2.COLORMAP_JET)
    overlay=cv2.cvtColor(overlay,cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1); plt.imshow(sample_rgb); plt.title('Input')
    plt.subplot(1,2,2); plt.imshow(sample_rgb); plt.imshow(overlay,alpha=.35); plt.title(f'Grad-CAM: {id_to_class[cls_idx]}')
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'gradcam_example.png'),dpi=180); plt.show()
except Exception as e:
    print("Grad-CAM skipped:",repr(e))

# Permutation importance for the trained compact fused classifier (feature-space explanation).
def permutation_importance(model,V,G,y,repeat=2):
    base=accuracy_score(y,model.predict((V,G),verbose=0).argmax(axis=1))
    rows=[]
    for modality,X in [('visual',V),('geometric',G)]:
        for j in range(X.shape[1]):
            drops=[]
            for _ in range(repeat):
                Xp=X.copy(); np.random.shuffle(Xp[:,j])
                predp=model.predict((Xp,G) if modality=='visual' else (V,Xp),verbose=0).argmax(axis=1)
                drops.append(base-accuracy_score(y,predp))
            rows.append((modality,j,float(np.mean(drops))))
    return pd.DataFrame(rows,columns=['modality','feature','accuracy_drop']).sort_values('accuracy_drop',ascending=False)
# Can be expensive for hundreds of PCA features; set RUN_PERMUTATION=False to skip.
RUN_PERMUTATION=False
if RUN_PERMUTATION:
    perm_df=permutation_importance(mats,Vte_p,Gte_p,y_test)
    perm_df.to_csv(os.path.join(OUTPUT_DIR,'permutation_importance.csv'),index=False)
    display(perm_df.head(20))

## 17. 3D embedding, sparsity and reliability visualizations

In [ ]:
# t-SNE on a bounded test subset to avoid excessive runtime.
n_vis=min(2000,len(Z_test))
if n_vis>=10:
    z3=TSNE(n_components=3,random_state=SEED,init='pca',learning_rate='auto').fit_transform(Z_test[:n_vis])
    fig=plt.figure(figsize=(9,7)); ax=fig.add_subplot(111,projection='3d')
    sc=ax.scatter(z3[:,0],z3[:,1],z3[:,2],c=y_test[:n_vis],s=8,cmap='tab20')
    ax.set_title('3D t-SNE — SLRF-Net embeddings'); ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2'); ax.set_zlabel('t-SNE 3')
    plt.colorbar(sc); plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'tsne_3d.png'),dpi=180); plt.show()

sparsity_ratio=float(np.mean(np.abs(slrf.proj.proj.numpy())<1e-3))
print("Projection sparsity ratio (|weight| < 1e-3):",sparsity_ratio)
plt.figure(figsize=(7,4)); plt.hist(wvis,bins=30,alpha=.7,label='Visual reliability')
plt.hist(wgeo,bins=30,alpha=.7,label='Geometric reliability')
plt.xlabel('Reliability weight'); plt.ylabel('Count'); plt.title('Adaptive Modality Reliability Distribution')
plt.legend(); plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'modality_reliability.png'),dpi=180); plt.show()

## 18. Cross-dataset evaluation interface

In [ ]:
def load_external_class_folder(root):
    d=discover_class_folder(root)
    if len(d)==0: raise ValueError("No class-folder images found at "+str(root))
    unknown=set(d.label)-set(class_to_id)
    if unknown: print("External labels not in primary training label map (excluded):",sorted(unknown))
    d=d[d.label.isin(class_to_id)].copy()
    d['y']=d.label.map(class_to_id).astype(int)
    return d

if EXTERNAL_TEST_ROOT and os.path.isdir(EXTERNAL_TEST_ROOT):
    ext_df=load_external_class_folder(EXTERNAL_TEST_ROOT)
    ext_ds=make_image_ds(ext_df,False)
    ext_logits=baseline_model.predict(ext_ds,verbose=0)
    ext_metrics,_,_,_=classification_metrics(ext_df.y.to_numpy(),ext_logits,'External MobileNet')
    print("Cross-dataset result:",ext_metrics)
else:
    print("External dataset not configured. Cross-dataset generalization cannot be claimed from the primary test set.")

## 19. Save reproducible outputs and final experiment summary

In [ ]:
# Save model weights / configs and fitted preprocessing objects.
baseline_model.save(os.path.join(OUTPUT_DIR,'mobilenetv3_baseline.keras'))
slrf.save(os.path.join(OUTPUT_DIR,'slrf_net.keras'))
mats.save(os.path.join(OUTPUT_DIR,'mats_fuse.keras'))
joblib.dump({'classes':classes,'class_to_id':class_to_id,'vis_scaler':vis_scaler,
             'geo_scaler':geo_scaler,'vis_pca':vis_pca,'geo_pca':geo_pca,
             'prototypes':prototypes,'temperature':temperature,'conformal_qhat':qhat,
             'thresholds':{'confidence':conf_thr,'entropy':ent_thr,'prototype_distance':dist_thr}},
            os.path.join(OUTPUT_DIR,'preprocessing_and_ucr_gate.joblib'))

summary={
    'dataset_root':DATA_ROOT,'n_train':len(train_df),'n_validation':len(val_df),'n_test':len(test_df),
    'n_classes':len(classes),'classes':classes,'temperature':temperature,'conformal_qhat':qhat,
    'conformal_test_coverage':float(coverage),'mean_prediction_set_size':float(set_sizes.mean()),
    'ucr_gate_acceptance_rate':float(accepted.mean()),'ucr_gate_selective_accuracy':float(selective_acc) if np.isfinite(selective_acc) else None,
    'projection_sparsity_ratio':sparsity_ratio,'ece_before':ece1,'ece_after':ece2,
    'sequence_data_available':bool(sequence_data_available),'output_dir':OUTPUT_DIR
}
with open(os.path.join(OUTPUT_DIR,'experiment_summary.json'),'w') as f: json.dump(summary,f,indent=2)
print(json.dumps(summary,indent=2))
print("All generated outputs:",sorted(os.listdir(OUTPUT_DIR)))

## Interpretation and research-reporting safeguards

1. Use the original subject-separated HaGRIDv2 partitions. The notebook intentionally refuses to invent a random train/test split when the supplied partition folders are missing.
2. A static image dataset can support image appearance and per-image geometric descriptors, but **cannot establish temporal motion recognition**. Supply the sequence manifest to train/evaluate the temporal branch.
3. The fallback geometric descriptor is an image-shape descriptor, not a hand-landmark estimate. For landmark-based claims, install a compatible MediaPipe version and verify the detector's landmark detection rate.
4. Conformal coverage is distribution-dependent and requires an independent calibration split. Report the calibration protocol and exchangeability limitations.
5. Edge latency depends on hardware. Report the measured device, batch size, warm-up, and runtime; do not claim target-device energy savings without a power measurement.
6. ANOVA/Tukey comparisons require repeated independent runs or folds for every compared method. Do not infer significance from one test score per model.
7. The requested comparisons to YOLOv13, GenENet, L-SHADE and GNN require their exact published definitions, implementations, and training protocol. This notebook does not fabricate those implementations.